In [1]:
# IMPORTS AND SETUP

import pandas as pd
import numpy as np
import yfinance as yf
import json
import time
import joblib
import chromadb
import os
import glob
import re
from tqdm.notebook import tqdm  
import ipywidgets as widgets
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sentence_transformers import SentenceTransformer
from textblob import TextBlob
import math
from datetime import datetime
import requests
import warnings

warnings.filterwarnings('ignore')

# Configuration
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "mistral"
SPLITS_DIR     = "../data/splits/"   # folder that holds aapl_YEAR*.csv files
OUTPUT_DIR     = "./"
CHECKPOINT_DIR = "./checkpoints/"

# INPUT_FILE and CURRENT_BATCH_KEY are set dynamically in Cell 2
INPUT_FILE        = "../data/splits/aapl_2023_batch15of16.csv"  # ← change per batch
CURRENT_BATCH_KEY = "aapl_2023_batch15of16"   
CURRENT_YEAR=2023
SKIP_LLM=False

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("✓ All imports successful")
print(f"✓ Checkpoint directory: {CHECKPOINT_DIR}")
print(f"✓ Splits directory    : {SPLITS_DIR}")


✓ All imports successful
✓ Checkpoint directory: ./checkpoints/
✓ Splits directory    : ../data/splits/


In [2]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # CELL 2: SELECT YEAR AND BATCH TO TRAIN
# # ═══════════════════════════════════════════════════════════════════════════════
# print("\n" + "="*80)
# print("SELECT YEAR AND BATCH TO TRAIN")
# print("="*80)

# # ── helper: discover batch files for a given year ────────────────────────────
# def get_batch_options(year):
#     """Return sorted list of (label, filepath) for the given year."""
#     # batched year:  aapl_2021_batch1of26.csv …
#     batched = sorted(glob.glob(os.path.join(SPLITS_DIR, f"aapl_{year}_batch*.csv")))
#     # single-file year:  aapl_2020.csv
#     single  = os.path.join(SPLITS_DIR, f"aapl_{year}.csv")
#     if batched:
#         return [(os.path.basename(p).replace(".csv","").replace(f"aapl_{year}_",""), p)
#                 for p in batched]
#     elif os.path.exists(single):
#         return [(f"{year}_single", single)]
#     else:
#         return [("(no file found)", None)]

# # ── widgets ──────────────────────────────────────────────────────────────────
# year_dropdown = widgets.Dropdown(
#     options=[2019, 2020, 2021, 2022, 2023, 2024],
#     value=2021,
#     description="Year:",
#     style={"description_width": "initial"},
#     layout=widgets.Layout(width="300px")
# )

# _initial_batches = get_batch_options(year_dropdown.value)
# batch_dropdown = widgets.Dropdown(
#     options=[(lbl, path) for lbl, path in _initial_batches],
#     description="Batch:",
#     style={"description_width": "initial"},
#     layout=widgets.Layout(width="420px")
# )

# skip_llm_checkbox = widgets.Checkbox(
#     value=False,
#     description="Skip LLM Extraction (for testing)",
#     indent=False
# )

# confirm_button = widgets.Button(
#     description="✅ Confirm Selection",
#     button_style="success",
#     layout=widgets.Layout(width="200px", height="35px")
# )

# status_output = widgets.Output()

# # ── state variables ───────────────────────────────────────────────────────────
# CURRENT_YEAR      = year_dropdown.value
# SKIP_LLM          = skip_llm_checkbox.value
# # INPUT_FILE and CURRENT_BATCH_KEY set on confirm

# # ── callbacks ────────────────────────────────────────────────────────────────
# def on_year_change(change):
#     """Refresh batch dropdown when year changes."""
#     global CURRENT_YEAR
#     CURRENT_YEAR = change["new"]
#     new_batches  = get_batch_options(CURRENT_YEAR)
#     batch_dropdown.options = [(lbl, path) for lbl, path in new_batches]
#     if batch_dropdown.options:
#         batch_dropdown.value = batch_dropdown.options[0][1]

# def on_skip_change(change):
#     global SKIP_LLM
#     SKIP_LLM = change["new"]

# def on_confirm_clicked(b):
#     global CURRENT_YEAR, SKIP_LLM, INPUT_FILE, CURRENT_BATCH_KEY
#     CURRENT_YEAR = year_dropdown.value
#     SKIP_LLM     = skip_llm_checkbox.value
#     INPUT_FILE   = batch_dropdown.value          # full path to the selected CSV

#     # Build a short key used for ALL checkpoint filenames
#     # e.g.  "2021_batch2of26"  or  "2021_single"  or  "2021"
#     selected_label = [lbl for lbl, path in batch_dropdown.options
#                       if path == INPUT_FILE][0]
#     CURRENT_BATCH_KEY = f"{CURRENT_YEAR}_{selected_label}"

#     with status_output:
#         status_output.clear_output(wait=True)
#         print(f"✓ Selected year      : {CURRENT_YEAR}")
#         print(f"✓ Selected batch     : {selected_label}")
#         print(f"✓ Input file         : {INPUT_FILE}")
#         print(f"✓ Checkpoint key     : {CURRENT_BATCH_KEY}")
#         print(f"  (checkpoints will be named like {CURRENT_BATCH_KEY}_aapl_extracted.csv)")
#         print(f"✓ Skip LLM           : {SKIP_LLM}")
#         print(f"\n✅ Ready! Now run the next cells.")

# # ── attach observers ─────────────────────────────────────────────────────────
# year_dropdown.observe(on_year_change, names="value")
# skip_llm_checkbox.observe(on_skip_change, names="value")
# confirm_button.on_click(on_confirm_clicked)

# # ── display ───────────────────────────────────────────────────────────────────
# display(year_dropdown)
# display(batch_dropdown)
# display(skip_llm_checkbox)
# display(confirm_button)
# display(status_output)


In [3]:
# UTILITY FUNCTIONS

def get_checkpoint_path(filename, batch_key=None):
    """
    Return full path for a checkpoint file.
    batch_key – e.g. "2021_batch2of26" or "2021_single".
    Falls back to CURRENT_BATCH_KEY when omitted.
    Model-level files (pkl) are global and never prefixed.
    """
    GLOBAL_FILES = {"impact_model.pkl", "feature_cols.pkl", "type_multipliers.pkl"}
    if filename in GLOBAL_FILES:
        return f"{CHECKPOINT_DIR}{filename}"
    key = batch_key if batch_key is not None else CURRENT_BATCH_KEY
    if key:
        return f"{CHECKPOINT_DIR}{key}_{filename}"
    return f"{CHECKPOINT_DIR}{filename}"

def load_previous_training_data():
    """Load all previously trained data from previous years"""
    all_data = []
    
    for file in sorted(os.listdir(CHECKPOINT_DIR)):
        if file.endswith('_training_complete.csv'):
            filepath = os.path.join(CHECKPOINT_DIR, file)
            try:
                df = pd.read_csv(filepath)
                all_data.append(df)
                print(f"  ✓ Loaded {len(df):,} rows from {file}")
            except Exception as e:
                print(f"  ✗ Error loading {file}: {e}")
    
    if all_data:
        combined = pd.concat(all_data, ignore_index=True)
        print(f"\n✓ Total accumulated data: {len(combined):,} rows")
        return combined
    else:
        print("✓ No previous training data found (first run)")
        return None

def load_previous_model():
    """Load previously trained model if it exists"""
    model_path = get_checkpoint_path('impact_model.pkl')
    feature_path = get_checkpoint_path('feature_cols.pkl')
    mult_path = get_checkpoint_path('type_multipliers.pkl')
    
    if os.path.exists(model_path) and os.path.exists(feature_path) and os.path.exists(mult_path):
        try:
            model = joblib.load(model_path)
            feature_cols = joblib.load(feature_path)
            type_multipliers = joblib.load(mult_path)
            print(f"✓ Loaded previous model from checkpoint")
            return model, feature_cols, type_multipliers
        except Exception as e:
            print(f"✗ Error loading previous model: {e}")
            return None, None, None
    else:
        print("✓ No previous model found (training from scratch)")
        return None, None, None

print("✓ Utility functions defined")

✓ Utility functions defined


In [4]:
# LOAD AND FILTER DATA

print("\n" + "="*80)
print(f"STEP 1: Load data for year {CURRENT_YEAR}")
print("="*80)

# Load CSV
if INPUT_FILE is None:
    raise RuntimeError("INPUT_FILE is not set. Run Cell 2 and click Confirm Selection first.")
print(f"✓ Reading: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

print(f"✓ Loaded {len(df):,} total articles")

# Filter for current year
start_of_year = pd.Timestamp(f'{CURRENT_YEAR}-01-01', tz='UTC')
end_of_year = pd.Timestamp(f'{CURRENT_YEAR}-12-31', tz='UTC')

df = df[(df['date'] >= start_of_year) &
        (df['date'] <= end_of_year)].copy()

print(f"✓ Articles for {CURRENT_YEAR}: {len(df):,}")

if len(df) == 0:
    print(f"✗ No articles found for year {CURRENT_YEAR}")
else:
    # Filter for AAPL
    aapl_variants = ['AAPL', 'AAPL.US', 'AAPL.BA', 'AAPL.MX', 'AAPL.SN', 'AAPL.NEO']
    df = df[df['symbols'].str.contains('|'.join(aapl_variants), case=False, na=False)].copy()
    
    print(f"✓ After AAPL filter: {len(df):,} articles")
    
    # Keep only needed columns
    df = df[['date', 'title', 'content']].copy()
    
    # Build text
    df['text'] = df.apply(
        lambda r: (f"{r['title']}. {r['content']}" if pd.notna(r['content']) else str(r['title']))[:2000],
        axis=1
    )
    
    df = df.dropna(subset=['text']).copy()
    df = df.reset_index(drop=True)
    
    print(f"✓ After cleanup: {len(df):,} rows")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Unique dates: {df['date'].dt.date.nunique():,}")



STEP 1: Load data for year 2023
✓ Reading: ../data/splits/aapl_2023_batch15of16.csv


FileNotFoundError: [Errno 2] No such file or directory: '../data/splits/aapl_2023_batch15of16.csv'

In [ ]:
# FETCH STOCK PRICES
print("\n" + "="*80)
print(f"STEP 2: Fetch stock prices for {CURRENT_YEAR}")
print("="*80)
start_date = df['date'].min() - pd.Timedelta(days=5)
end_date = df['date'].max() + pd.Timedelta(days=10)
print(f"Fetching prices from {start_date.date()} to {end_date.date()}...")
prices = yf.download("AAPL", start=start_date, end=end_date, progress=False)
prices.reset_index(inplace=True)
print(f"✓ Fetched {len(prices):,} trading days")

# MultiIndex columns
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = ['_'.join(col).strip('_') for col in prices.columns.values]

# Find date column
date_col = None
for col in prices.columns:
    if 'date' in col.lower():
        date_col = col
        break
if date_col:
    prices[date_col] = pd.to_datetime(prices[date_col])
    if prices[date_col].dt.tz is not None:
        prices[date_col] = prices[date_col].dt.tz_localize(None)
    prices = prices.rename(columns={date_col: 'Date'})
else:
    prices.index = pd.to_datetime(prices.index)
    if prices.index.tz is not None:
        prices.index = prices.index.tz_localize(None)
    prices = prices.reset_index()
    if 'index' in prices.columns:
        prices = prices.rename(columns={'index': 'Date'})

# Keep only Close prices
close_col = None
for col in prices.columns:
    if 'close' in col.lower():          
        close_col = col
        break

prices = prices[['Date', close_col]].copy()
prices.columns = ['Date', 'Close']

prices = prices.sort_values('Date').reset_index(drop=True)
prices['Date'] = prices['Date'].dt.tz_localize(None) if prices['Date'].dt.tz is not None else prices['Date']
print(f"✓ Price data: {len(prices):,} days")



In [ ]:
# CELL 6: STEP 3 - CALCULATE PRICE OUTCOMES

print("\n" + "="*80)
print(f"STEP 3: Calculate price outcomes for {CURRENT_BATCH_KEY}")
print("="*80)

price_lookup = {}
for idx, row in prices.iterrows():
    date_key = row['Date'].date()
    price_lookup[date_key] = {'idx': idx, 'close': row['Close']}

def calc_price_change(date, price_dict, price_df, window):
    try:
        date = pd.Timestamp(date)
        if date.tz is not None:
            date = date.tz_localize(None)
        
        date_key = date.date()
        if date_key not in price_dict:
            return np.nan
        
        matching_idx = price_dict[date_key]['idx']
        if matching_idx + window >= len(price_df):
            return np.nan
        
        today_close = float(price_df.iloc[matching_idx]['Close'])
        future_close = float(price_df.iloc[matching_idx + window]['Close'])
        
        pct_change = (future_close - today_close) / today_close * 100
        return float(pct_change)
    except:
        return np.nan

print("Calculating T+1, T+3, T+5 price changes...")

# tqdm for progress bar
pct_change_T1_list = []
pct_change_T3_list = []
pct_change_T5_list = []

for date in tqdm(df['date'].values, desc="Calculating price changes"):
    pct_T1 = calc_price_change(date, price_lookup, prices, 1)
    pct_T3 = calc_price_change(date, price_lookup, prices, 3)
    pct_T5 = calc_price_change(date, price_lookup, prices, 5)
    
    pct_change_T1_list.append(pct_T1)
    pct_change_T3_list.append(pct_T3)
    pct_change_T5_list.append(pct_T5)

df['pct_change_T1'] = pd.Series(pct_change_T1_list, dtype='float64')
df['pct_change_T3'] = pd.Series(pct_change_T3_list, dtype='float64')
df['pct_change_T5'] = pd.Series(pct_change_T5_list, dtype='float64')

before = len(df)
df = df.dropna(subset=['pct_change_T1']).copy()
df = df.reset_index(drop=True)

print(f"✓ Outcomes calculated: {len(df):,} rows (dropped {before - len(df):,})")

# Create direction labels
direction_list = []
for val in df['pct_change_T1'].values:
    val = float(val)
    if val > 0.5:
        direction_list.append('HIGH')
    elif val < -0.5:
        direction_list.append('LOW')
    else:
        direction_list.append('NEUTRAL')

df['direction_T1'] = direction_list

print(f"✓ Outcome distribution:")
print(f"  HIGH:    {(df['direction_T1'] == 'HIGH').sum():,}")
print(f"  LOW:     {(df['direction_T1'] == 'LOW').sum():,}")
print(f"  NEUTRAL: {(df['direction_T1'] == 'NEUTRAL').sum():,}")

# Save checkpoint
df.to_csv(get_checkpoint_path('aapl_with_outcomes.csv', CURRENT_BATCH_KEY), index=False)


In [ ]:
# CELL 7: STEP 4 - COMPUTE SENTIMENT

print("\n" + "="*80)
print(f"STEP 4: Compute sentiment for {CURRENT_BATCH_KEY}")
print("="*80)

print("Computing sentiment scores with TextBlob...")

sentiment_scores = []
for text in tqdm(df['text'].values, desc="Computing sentiment"):
    try:
        blob = TextBlob(str(text))
        polarity = blob.sentiment.polarity
        sentiment_scores.append(polarity)
    except:
        sentiment_scores.append(0.0)

df['sentiment_score'] = sentiment_scores

print(f"✓ Sentiment computed:")
print(f"  Mean:  {df['sentiment_score'].mean():.3f}")
print(f"  Min:   {df['sentiment_score'].min():.3f}")
print(f"  Max:   {df['sentiment_score'].max():.3f}")

df.to_csv(get_checkpoint_path('aapl_with_sentiment.csv', CURRENT_BATCH_KEY), index=False)

In [ ]:
# LLM EVENT EXTRACTION (GROQ)
from groq import Groq
import os
import time

print("\n" + "="*80)
print(f"STEP 5: LLM Event Extraction for {CURRENT_BATCH_KEY}")
print("="*80)

GROQ_API_KEY = "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"
GROQ_MODEL   = "llama-3.1-8b-instant"

groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Groq free tier: 30 requests/min
REQUESTS_PER_MINUTE = 28
DELAY_BETWEEN_CALLS = 60 / REQUESTS_PER_MINUTE  # ~2.1s

llm_checkpoint = get_checkpoint_path('aapl_extracted.csv', CURRENT_BATCH_KEY)
print("llm checkpoint path is", llm_checkpoint)

if os.path.exists(llm_checkpoint):
    print(f"✓ Checkpoint found — loading from disk, skipping LLM extraction")
    df = pd.read_csv(llm_checkpoint)
    print(f"✓ Loaded {len(df):,} rows from checkpoint")
    print(f"Event type distribution:")
    print(df['event_type'].value_counts())

elif SKIP_LLM:
    print("⏭️  Skipping LLM extraction (test mode)")
    df['event_type']       = 'general'
    df['severity']         = 5.0
    df['affected_product'] = 'General'
    print("✓ Default values assigned")
    print(f"Event type distribution:")
    print(df['event_type'].value_counts())
    df.to_csv(llm_checkpoint, index=False)
    print(f"✓ Saved to checkpoint")

else:
    # Prompt
    PROMPT_SYSTEM = "You are a financial news analyzer. Return ONLY valid JSON, no explanation, no markdown."

    PROMPT_USER = """Extract event information from this Apple news article.
Return ONLY this exact JSON structure with no other text:
{{
  "event_type": one of [earnings_beat, earnings_miss, product_launch, legal, supply_chain, analyst_upgrade, analyst_downgrade, ceo_statement, regulatory, general],
  "severity": float from 0.0 to 10.0,
  "affected_product": one of [iPhone, Mac, Services, App_Store, Vision_Pro, General]
}}

Article: {article_text}"""

    # Extract function
    def extract_with_groq(article_text, retries=3):
        for attempt in range(retries):
            try:
                response = groq_client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=[
                        {"role": "system", "content": PROMPT_SYSTEM},
                        {"role": "user",   "content": PROMPT_USER.format(article_text=article_text[:1500])}
                    ],
                    temperature=0.3,
                    max_tokens=100,
                )
                result_text = response.choices[0].message.content.strip()

                try:
                    return json.loads(result_text)
                except json.JSONDecodeError:
                    start_idx = result_text.find('{')
                    end_idx   = result_text.rfind('}') + 1
                    if start_idx != -1 and end_idx > start_idx:
                        return json.loads(result_text[start_idx:end_idx])

            except Exception as e:
                error_msg = str(e)

                # Rate limit — wait and retry
                if 'rate_limit' in error_msg.lower() or '429' in error_msg:
                    wait_time = 60 * (attempt + 1)
                    print(f"\n  ⚠ Rate limit hit — waiting {wait_time}s before retry {attempt+1}/{retries}")
                    time.sleep(wait_time)
                    continue

                print(f"\n  ⚠ Groq error (attempt {attempt+1}): {error_msg[:100]}")
                time.sleep(5)

        return {"event_type": "general", "severity": 5.0, "affected_product": "General"}

    # Run Extraction
    total       = len(df)
    est_minutes = total * DELAY_BETWEEN_CALLS / 60

    print(f"✓ Groq client ready — model: {GROQ_MODEL}")
    print(f"  Articles to process : {total:,}")
    print(f"  Rate limit delay    : {DELAY_BETWEEN_CALLS:.1f}s between calls")
    print(f"  Estimated time      : {est_minutes:.1f} minutes")

    extracted_results = [None] * total
    last_save_idx     = 0

    df = df.reset_index(drop=True)

    with tqdm(total=total, desc="Groq Extraction") as pbar:
        for idx, row in df.iterrows():
            extracted_results[idx] = extract_with_groq(row['text'])

            # Rate limit delay
            time.sleep(DELAY_BETWEEN_CALLS)
            pbar.update(1)

            # Auto-save every 50 articles in case of crash
            if (idx + 1) % 50 == 0:
                completed_extracted = [r for r in extracted_results if r is not None]
                completed_df        = df.iloc[:len(completed_extracted)].reset_index(drop=True)

                if completed_extracted:
                    temp_df = pd.concat(
                        [completed_df, pd.DataFrame(completed_extracted)],
                        axis=1
                    )
                    temp_df.to_csv(llm_checkpoint + '.tmp', index=False)
                    last_save_idx = idx + 1
                    print(f"\n Auto-saved {len(completed_extracted):,} / {total:,} articles")

    extracted_results = [
        r if r is not None else
        {"event_type": "general", "severity": 5.0, "affected_product": "General"}
        for r in extracted_results
    ]

    extracted_df = pd.DataFrame(extracted_results)
    df = pd.concat([df.reset_index(drop=True), extracted_df], axis=1)

    print(f"\n✓ Groq extraction complete!")
    print(f"Event type distribution:")
    print(df['event_type'].value_counts())

    df.to_csv(llm_checkpoint, index=False)
    print(f"✓ Saved checkpoint to {llm_checkpoint}")

    # Clean up temp file
    if os.path.exists(llm_checkpoint + '.tmp'):
        os.remove(llm_checkpoint + '.tmp')
        print(f"✓ Temp file cleaned up")

In [ ]:
# CELL 9: STEP 6 - FEATURE ENGINEERING
import numpy as np

print("\n" + "="*80)
print(f"STEP 6: Feature Engineering for {CURRENT_YEAR}")
print("="*80)

def calc_impact_score(row):
    pct = float(row['pct_change_T1'])
    price_comp = min(abs(pct) / 5.0, 1.0)
    vol_comp = 0.3
    idio_comp = 0.2
    return round(0.5 * price_comp + 0.3 * vol_comp + 0.2 * idio_comp, 3)

df['impact_score'] = df.apply(calc_impact_score, axis=1)

VALID_PRODUCTS = ['iPhone', 'Mac', 'Services', 'App_Store', 'Vision_Pro', 'General']
VALID_EVENTS   = ['earnings_beat', 'earnings_miss', 'product_launch', 'legal',
                  'supply_chain', 'analyst_upgrade', 'analyst_downgrade',
                  'ceo_statement', 'regulatory', 'general']

def clean_product(val):
    if isinstance(val, (list, np.ndarray)):
        for product in VALID_PRODUCTS:
            if any(product.lower() in str(v).lower() for v in val):
                return product
        return 'General'
    if pd.isna(val):
        return 'General'
    val_str = str(val).strip()
    if val_str.startswith('['):
        for product in VALID_PRODUCTS:
            if product.lower() in val_str.lower():
                return product
        return 'General'
    for product in VALID_PRODUCTS:
        if product.lower() == val_str.lower():
            return product
    return 'General'

def clean_event_type(val):
    if isinstance(val, (list, np.ndarray)):
        for evt in VALID_EVENTS:
            if any(evt in str(v).lower() for v in val):
                return evt
        return 'general'
    if pd.isna(val):
        return 'general'
    val_str = str(val).strip()
    if val_str.startswith('['):
        for evt in VALID_EVENTS:
            if evt in val_str.lower():
                return evt
        return 'general'
    return val_str if val_str in VALID_EVENTS else 'general'

df['affected_product'] = df['affected_product'].apply(clean_product)
df['event_type']       = df['event_type'].apply(clean_event_type)

print(f"✓ Cleaned event_type values:       {df['event_type'].unique().tolist()}")
print(f"✓ Cleaned affected_product values: {df['affected_product'].unique().tolist()}")

event_types = sorted(df['event_type'].unique())
for evt in event_types:
    df[f'evt_{evt}'] = (df['event_type'] == evt).astype(int)

products = sorted(df['affected_product'].unique())
for prod in products:
    df[f'prod_{prod}'] = (df['affected_product'] == prod).astype(int)

# Event type multipliers
type_multipliers = {}
for evt_type in event_types:
    avg_impact = df[df['event_type'] == evt_type]['impact_score'].mean()
    type_multipliers[evt_type] = round(1.0 + avg_impact, 3)

print(f"✓ Event type multipliers:")
for evt, mult in sorted(type_multipliers.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {evt:20s}: {mult:.3f}")

FEATURE_COLS = (
    ['sentiment_score', 'severity'] +
    [f'evt_{evt}' for evt in event_types] +
    [f'prod_{prod}' for prod in products]
)
print(f"✓ Total features: {len(FEATURE_COLS)}")

bad_cols = [c for c in FEATURE_COLS if any(x in c for x in ['[', ']', '<', '>'])]
if bad_cols:
    print(f"⚠ Bad column names still found: {bad_cols}")
else:
    print(f"✓ All feature column names are clean")

df.to_csv(get_checkpoint_path('aapl_training_ready.csv', CURRENT_BATCH_KEY), index=False)
print(f"✓ Saved training-ready data to checkpoint")

In [ ]:
bad_cols = [c for c in FEATURE_COLS if any(x in c for x in ['[', ']', '<', '>'])]
print("Bad columns:", bad_cols)

In [ ]:
# INCREMENTAL MODEL TRAINING
print("\n" + "="*80)
print(f"STEP 7: Train XGBoost (Incremental Learning)")
print("="*80)

def align_to_feature_cols(frame, feature_cols):
    """Add any missing one-hot columns (filled with 0) and return only feature_cols."""
    for col in feature_cols:
        if col not in frame.columns:
            frame[col] = 0
    return frame[feature_cols]

def build_full_frame(frame):
    carry_cols = ['impact_score', 'event_type', 'affected_product',
                  'sentiment_score', 'severity']
    feat_frame  = align_to_feature_cols(frame.copy(), FEATURE_COLS)
    carry_frame = frame[[c for c in carry_cols if c in frame.columns]].reset_index(drop=True)
    return pd.concat([carry_frame, feat_frame.reset_index(drop=True)], axis=1)

def make_X(frame, feature_cols):
    """
    Extract feature matrix, deduplicate columns, and cast to float32.
    Duplicate column names cause pandas to return a DataFrame slice instead
    of a Series when XGBoost iterates columns — triggering AttributeError on .dtype.
    """
    X = frame[feature_cols].fillna(0)
    # Drop duplicate columns (keep first occurrence)
    X = X.loc[:, ~X.columns.duplicated()]
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
    return X.astype('float32')

# Load previous data
print("Loading previous training data...")
previous_data = load_previous_training_data()

if previous_data is not None:
    print(f"\n✓ Merging {len(df):,} new rows with {len(previous_data):,} previous rows")
    df_merged   = pd.concat([build_full_frame(previous_data),
                              build_full_frame(df)],
                             ignore_index=True)
    print(f"✓ Total training data: {len(df_merged):,} rows")
    training_df = df_merged
else:
    print(f"\n✓ First training session: {len(df):,} rows")
    training_df = df

# check before training
dup_cols = [c for c in FEATURE_COLS if FEATURE_COLS.count(c) > 1]
if dup_cols:
    print(f"⚠ Duplicate feature names detected and will be removed: {set(dup_cols)}")
    FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))   # deduplicate, preserve order

X = make_X(training_df, FEATURE_COLS)
y = training_df['impact_score'].astype('float32')

split_idx = int(0.8 * len(training_df))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {len(FEATURE_COLS)}")

model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=30,
)

print("Training XGBoost...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)

train_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae  = mean_absolute_error(y_test,  model.predict(X_test))
print(f"✓ Train MAE: {train_mae:.4f}")
print(f"✓ Test MAE:  {test_mae:.4f}")

joblib.dump(model,            get_checkpoint_path('impact_model.pkl'))
joblib.dump(FEATURE_COLS,     get_checkpoint_path('feature_cols.pkl'))
joblib.dump(type_multipliers, get_checkpoint_path('type_multipliers.pkl'))
print(f"✓ Model saved to checkpoint")

In [ ]:
# STORE IN CHROMADB
print("\n" + "="*80)
print(f"STEP 8: Store in ChromaDB (Incremental)")
print("="*80)

df['predicted_impact_score'] = model.predict(X[:len(df)])

df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)

def compute_event_weight(row):
    type_mult = type_multipliers.get(row['event_type'], 1.0)
    base = (1.0 + row['predicted_impact_score']) * type_mult
    days_ago = (datetime.now() - pd.Timestamp(row['date'])).days
    time_decay = math.exp(-0.693 * days_ago / 365)
    return round(base * time_decay, 4)

df['event_weight'] = df.apply(compute_event_weight, axis=1)
print(f"✓ Event weights: Mean={df['event_weight'].mean():.3f}")

df.to_csv(get_checkpoint_path('aapl_training_complete.csv', CURRENT_BATCH_KEY), index=False)

# Store in ChromaDB
NEW_CHROMA_PATH = f"{OUTPUT_DIR}aapl_memory_v2"
chroma_client = chromadb.PersistentClient(path=NEW_CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name="aapl_events",
    metadata={"hnsw:space": "cosine"}
)
print(f"✓ Using ChromaDB at: {NEW_CHROMA_PATH}")

print("Loading FinBERT embeddings...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")   # faster, smaller, very reliable

print(f"Storing {len(df):,} events in ChromaDB...")

stored = 0
skipped = 0

for idx, row in enumerate(tqdm(df.iterrows(), total=len(df), desc="Storing in ChromaDB")):
    try:
        embedding = encoder.encode(row[1]['text']).tolist()
        event_id  = f"aapl_{CURRENT_YEAR}_{str(row[1]['date'].date())}_{idx}"

        existing = collection.get(ids=[event_id])
        if existing['ids']:
            skipped += 1
            continue

        collection.add(
            ids=[event_id],
            embeddings=[embedding],
            documents=[row[1]['text']],
            metadatas=[{
                'year':             int(CURRENT_YEAR),
                'date':             str(row[1]['date'].date()),
                'title':            str(row[1]['title'])[:500],
                'event_type':       str(row[1]['event_type']),
                'severity':         float(row[1]['severity']),
                'affected_product': str(row[1]['affected_product']),
                'sentiment_score':  float(row[1]['sentiment_score']),
                'impact_score':     float(row[1]['predicted_impact_score']),
                'event_weight':     float(row[1]['event_weight']),
                'direction':        str(row[1]['direction_T1']),
            }]
        )
        stored += 1

    except Exception as e:
        print(f"  ⚠ Row {idx} failed: {e}")
        skipped += 1
        continue

print(f"✓ Stored:  {stored:,} new events")
print(f"✓ Skipped: {skipped:,} (duplicates or errors)")
print(f"✓ Total in collection: {collection.count():,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12: FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print(f"✓ SESSION COMPLETE - YEAR {CURRENT_YEAR}")
print("="*80)
print(f"Articles processed: {len(df):,}")
print(f"Test MAE: {test_mae:.4f}")
print(f"\nCheckpoint files saved in {CHECKPOINT_DIR}:")
print(f"  ✓ {CURRENT_BATCH_KEY}_aapl_training_complete.csv")
print(f"  ✓ impact_model.pkl")
print(f"  ✓ ChromaDB updated")

# Check total progress
total_files = len([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('_training_complete.csv')])
print(f"\nTotal sessions completed: {total_files}")

next_year = CURRENT_YEAR + 1
print(f"\nNext steps:")
print(f"  1. Wait 5 hours between sessions")
print(f"  2. Run CELLS 2-12 again — pick the next batch, or year={next_year} when all batches done")
print(f"  3. Model will automatically merge and improve!")
print("="*80)

In [ ]:
import chromadb
OUTPUT_DIR='./'
client = chromadb.PersistentClient(path=f"{OUTPUT_DIR}aapl_memory_v2")

# See all collections
collections = client.list_collections()
print("Collections:", collections)

for col_info in collections:
    collection = client.get_collection(col_info.name)
    print(f"\n{'='*50}")
    print(f"Collection: {col_info.name}")
    print(f"Total items stored: {collection.count()}")
    
    # Preview first 10
    peek = collection.peek(10)
    print("\n--- IDs ---")
    print(peek['ids'])
    print("\n--- Documents (text chunks) ---")
    for doc in peek['documents']:
        print(doc[:200], "...")   # first 200 chars
    print("\n--- Metadata ---")
    for meta in peek['metadatas']:
        print(meta)